# DMA2 — Openings as Memes: Analysis
**Spec:** `MASTER.md` · **process:** `ROADMAP.md` · **state:** `notes/decisions.md`

Rules of this notebook:
1. Every section is **self-contained**: it reads from the DB / `exports/` files, never from another section's variables. The only shared cell is *Constants & helpers*.
2. Anything exploratory goes in **Scratch** at the bottom — never above it.
3. Before committing or putting any number on a slide: **Kernel → Restart & Run All** must pass clean.

In [ ]:
# ── Constants & helpers — the ONLY cell other sections may depend on ──
# Every value here is mirrored in notes/decisions.md. Change in both places or not at all.
from pathlib import Path
import sqlite3, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd() if (Path.cwd() / "db").exists() else Path.cwd().parent
if not (ROOT / "db").exists():
    print("WARNING: ROOT looks wrong ->", ROOT)
DB      = ROOT / "db" / "chess.sqlite"
EXPORTS = ROOT / "exports"
FIGURES = ROOT / "figures"
RESULTS = EXPORTS / "results.md"

# ── Locked parameters (sync with decisions.md) ──
QG_EVENT,  QG_POST  = "2020-10-23", "2020-11"
BC_EVENT,  BC_POST  = "2021-03-??", "2021-03"   # TODO: verify exact Nakamura-Carlsen date, then lock
BASELINE_YEAR       = "2019"
TRIGGER_WINDOW      = ("2020-07", "2021-12")
MIN_GAMES_2019      = 5000
MIN_CLOSE_HIGH      = 200
N_PLACEBO           = 20
FOLD_VIRAL          = 4.0
PEAK_FLOOR          = 2e-4

plt.rcParams.update({"font.size": 15, "figure.figsize": (9, 5),
                     "figure.dpi": 110, "savefig.bbox": "tight"})

def q(sql, params=()):
    """Run SQL against the frozen agg table, return a DataFrame."""
    with sqlite3.connect(DB) as conn:
        return pd.read_sql_query(sql, conn, params=params)

def save_fig(name):
    FIGURES.mkdir(exist_ok=True)
    p = FIGURES / f"{name}.png"
    plt.savefig(p)
    print("saved", p)

def log_result(section, text):
    """Append headline numbers to exports/results.md (the audit in Phase 7 checks against this)."""
    EXPORTS.mkdir(exist_ok=True)
    with open(RESULTS, "a") as f:
        f.write(f"\n## {section} ({datetime.date.today().isoformat()})\n{text}\n")
    print(f"[results.md] {section} logged")

print("ROOT =", ROOT, "| DB exists:", DB.exists())

In [ ]:
# ── Smoke test (works once Phase 3 / build_db.py is done) ──
monthly = q("SELECT month, SUM(n) AS games FROM agg GROUP BY month ORDER BY month")
print(monthly.tail(3))
monthly.set_index("month")["games"].plot(title="Kept blitz games per month")
plt.xticks(rotation=90); plt.show()

In [ ]:
# ── Focal-series eyeball: the three sanity signals ──
# Expect: QG jump from 2020-11, Bongcloud spike around 2021-03, Stafford elevated through 2020-21.
focal = {"Queen's Gambit": "Queen's Gambit%", "Bongcloud": "%Bongcloud%", "Stafford": "%Stafford%"}
tot = q("SELECT month, SUM(n) AS t FROM agg GROUP BY month")
for label, pat in focal.items():
    s = q("SELECT month, SUM(n) AS g FROM agg WHERE opening LIKE ? GROUP BY month", (pat,))
    s = s.merge(tot, on="month")
    plt.plot(s["month"], s["g"] / s["t"], label=label)
plt.legend(); plt.xticks(rotation=90); plt.title("Share of all games per month"); plt.show()

## Phase 4 — Features → PCA → H2 (centerpiece)
Cheap-model session: paste `MASTER.md` + the Phase 4 card + the *Constants & helpers* cell, and ask for **Jupyter cells, not a script**.
**Contract:** read via `q()` / `exports/features.csv`; save PC1 scores to `exports/pc_scores.csv`; figures via `save_fig("h2_loadings")`, `save_fig("h2_scatter")`; headline numbers via `log_result("H2", ...)`. No variables from any other section.

In [ ]:
# Phase 4 cells go here

## Phase 5 — H1 structural breaks + placebos
Same procedure with the Phase 5 card. Verify and lock `BC_EVENT` in the constants cell **and** decisions.md first.
**Contract:** figures `h1_qg`, `h1_bongcloud` (time series + fitted pre/post lines + placebo band); `log_result("H1", ...)`. Reads only `q()` / exports.

In [ ]:
# Phase 5 cells go here

## Phase 6 — H4 decay half-lives (+ optional k-means)
Same procedure with the Phase 6 card.
**Contract:** PC1 comes from `exports/pc_scores.csv` (a file Phase 4 wrote — never from kernel memory); figure `h4_decay` (+ `k3_halflives` if the optional part runs); `log_result("H4", ...)`.

In [ ]:
# Phase 6 cells go here

## Scratch — exploration only below this line
Nothing above may depend on anything down here. Delete freely.

In [ ]:
# scratch